# Imports

In [9]:
from filters import * 

import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 


from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.base import clone
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

from scipy.special import betaln, logsumexp
from scipy.optimize import minimize

In [10]:
from pathlib import Path

def load_clean_dataset(dataset_name, data_root="./dataset/data_base"):
    """
    Load a clean dataset from the local folder structure.

    Expected file:
        ./dataset/data_base/<dataset_name>/<dataset_name>-<dataset_name>-cc.dat
    """
    path = Path(data_root) / dataset_name / f"{dataset_name}-{dataset_name}-cc.dat"
    if not path.exists():
        raise FileNotFoundError(f"Dataset file not found: {path}")

    lines = path.read_text(encoding="latin-1").splitlines()
    data_idx = next(i for i, line in enumerate(lines) if line.strip().lower() == "@data")

    rows = [line.strip() for line in lines[data_idx + 1:] if line.strip() and not line.strip().startswith("%")]

    X = []
    y = []
    for row in rows:
        parts = [p.strip() for p in row.split(",")]
        X.append([float(v) for v in parts[:-1]])
        y.append(parts[-1])

    return np.asarray(X, dtype=float), np.asarray(y)

In [11]:
class BetaMixture2:
    """
    Mezcla no supervisada de 2 distribuciones Beta para datos en [0, 1].

    Ajusta:
        p(x) = pi_0 * Beta(a_0, b_0) + pi_1 * Beta(a_1, b_1)

    mediante EM:
        - E-step: calcula responsabilidades
        - M-step: reestima pesos y parámetros Beta por máxima verosimilitud ponderada
    """

    def __init__(
        self,
        max_iter=500,
        tol=1e-6,
        n_init=10,
        eps=1e-6,
        random_state=None,
        verbose=False,
    ):
        self.max_iter = max_iter
        self.tol = tol
        self.n_init = n_init
        self.eps = eps
        self.random_state = random_state
        self.verbose = verbose

    @staticmethod
    def _beta_logpdf(x, a, b):
        return (a - 1) * np.log(x) + (b - 1) * np.log1p(-x) - betaln(a, b)

    def _weighted_moments_init(self, x, w):
        """
        Inicialización aproximada de alpha y beta por momentos ponderados.
        """
        w = np.asarray(w, dtype=float)
        w = w / np.sum(w)

        mean = np.sum(w * x)
        var = np.sum(w * (x - mean) ** 2)

        # Evitar varianzas degeneradas
        var = max(var, 1e-6)

        common = mean * (1 - mean) / var - 1

        if common <= 0:
            # Fallback razonable si los momentos no dan una Beta válida
            a, b = 1.0, 1.0
        else:
            a = mean * common
            b = (1 - mean) * common

        a = max(a, 1e-3)
        b = max(b, 1e-3)

        return a, b

    def _fit_weighted_beta(self, x, w, a_init, b_init):
        """
        Ajuste de una Beta por máxima verosimilitud ponderada.
        Optimiza sobre log(alpha), log(beta) para garantizar positividad.
        """
        w = np.asarray(w, dtype=float)
        w_sum = np.sum(w)

        if w_sum <= 1e-12:
            return a_init, b_init

        w = w / w_sum

        def objective(theta):
            log_a, log_b = theta
            a = np.exp(log_a)
            b = np.exp(log_b)

            logpdf = self._beta_logpdf(x, a, b)
            return -np.sum(w * logpdf)

        theta0 = np.log([a_init, b_init])

        result = minimize(
            objective,
            theta0,
            method="L-BFGS-B",
            bounds=[(-10, 10), (-10, 10)],
        )

        if not result.success:
            return a_init, b_init

        a, b = np.exp(result.x)
        return float(a), float(b)

    def _single_fit(self, x, rng, init_id):
        n = len(x)

        # Inicialización de responsabilidades
        resp = np.zeros((n, 2))

        if init_id == 0:
            # Inicialización determinista por mediana
            median = np.median(x)
            resp[:, 0] = x <= median
            resp[:, 1] = x > median

            # Evitar componentes vacías
            resp = resp + 1e-3
            resp = resp / resp.sum(axis=1, keepdims=True)
        else:
            # Inicialización aleatoria suave
            r = rng.uniform(0.1, 0.9, size=n)
            resp[:, 0] = r
            resp[:, 1] = 1 - r

        pis = resp.mean(axis=0)

        params = []
        for k in range(2):
            a, b = self._weighted_moments_init(x, resp[:, k])
            params.append([a, b])

        params = np.asarray(params, dtype=float)

        prev_ll = -np.inf

        for it in range(self.max_iter):
            # E-step
            log_prob = np.zeros((n, 2))

            for k in range(2):
                a, b = params[k]
                log_prob[:, k] = np.log(pis[k] + 1e-12) + self._beta_logpdf(x, a, b)

            log_norm = logsumexp(log_prob, axis=1)
            ll = np.sum(log_norm)

            resp = np.exp(log_prob - log_norm[:, None])

            # M-step
            pis = resp.mean(axis=0)

            for k in range(2):
                a_old, b_old = params[k]
                a_new, b_new = self._fit_weighted_beta(x, resp[:, k], a_old, b_old)
                params[k] = [a_new, b_new]

            if self.verbose:
                print(f"Init {init_id}, iter {it}, loglik={ll:.6f}")

            if np.abs(ll - prev_ll) < self.tol:
                break

            prev_ll = ll

        return {
            "loglik": ll,
            "pis": pis,
            "params": params,
            "resp": resp,
            "n_iter": it + 1,
        }

    def fit(self, x):
        x = np.asarray(x, dtype=float).ravel()

        if np.any(x < 0) or np.any(x > 1):
            raise ValueError("Todos los valores deben estar en el intervalo [0, 1].")

        # La Beta estándar está definida en (0, 1), no exactamente en 0 o 1
        x = np.clip(x, self.eps, 1 - self.eps)

        self.x_fit_ = x

        rng = np.random.default_rng(self.random_state)

        best = None

        for init_id in range(self.n_init):
            result = self._single_fit(x, rng, init_id)

            if best is None or result["loglik"] > best["loglik"]:
                best = result

        self.weights_ = best["pis"]
        self.params_ = best["params"]
        self.responsibilities_ = best["resp"]
        self.loglik_ = best["loglik"]
        self.n_iter_ = best["n_iter"]

        # Ordenar componentes por media creciente
        means = self.params_[:, 0] / (self.params_[:, 0] + self.params_[:, 1])
        order = np.argsort(means)

        self.weights_ = self.weights_[order]
        self.params_ = self.params_[order]
        self.responsibilities_ = self.responsibilities_[:, order]
        self.means_ = means[order]

        return self

    def predict_proba(self, x):
        x = np.asarray(x, dtype=float).ravel()
        x = np.clip(x, self.eps, 1 - self.eps)

        n = len(x)
        log_prob = np.zeros((n, 2))

        for k in range(2):
            a, b = self.params_[k]
            log_prob[:, k] = np.log(self.weights_[k] + 1e-12) + self._beta_logpdf(x, a, b)

        log_norm = logsumexp(log_prob, axis=1)
        resp = np.exp(log_prob - log_norm[:, None])

        return resp

    def predict(self, x):
        return np.argmax(self.predict_proba(x), axis=1)

    def score_samples(self, x):
        """
        Log densidad de la mezcla en cada punto.
        """
        x = np.asarray(x, dtype=float).ravel()
        x = np.clip(x, self.eps, 1 - self.eps)

        n = len(x)
        log_prob = np.zeros((n, 2))

        for k in range(2):
            a, b = self.params_[k]
            log_prob[:, k] = np.log(self.weights_[k] + 1e-12) + self._beta_logpdf(x, a, b)

        return logsumexp(log_prob, axis=1)

# Testing

In [ ]:
datasets = ["yeast"]
rs = 33

for dataset in datasets:

    # Load dataset
    X, y = load_clean_dataset(dataset)

    # Initilize classifier
    clf = LogisticRegression()

    # Initialize noise filter
    nf = ENNProbFilter(
    n_neighbors=9,
    n_jobs=-1,
    action= "detect"
    )

    # Perform 5CV
    for tr_idx, val_idx in StratifiedKFold(n_splits=5).split(X,y):
        # Preprocess train_set
        sc = StandardScaler()
        X_sc = sc.fit_transform(X[tr_idx])
        
        # Fit filter
        nf.fit(X_sc, y[tr_idx])

        
